# 02 · EDA y supuestos: trayectos de mañana

**Pregunta de este notebook:** antes de calcular distancias de Mahalanobis, ¿qué variables conviene usar y qué supuestos se cumplen?

Se trabaja solo con los *trayectos de mañana* (≈13,6 km, la misma ruta casi todos los días), que son el único grupo con muestra suficiente para estimar una matriz de covarianza. Las salidas largas se tratan aparte.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from stravamaha import plots

pd.options.display.float_format = "{:.3f}".format
%matplotlib inline

## 1. Muestra

In [ ]:
d = pd.read_csv(RAIZ / "data/processed/salidas.csv")
m = d[d["ruta"] == "trayecto_manana"].copy()

print(f"n = {len(m)} salidas | {m['fecha'].min()} -> {m['fecha'].max()}")
m.head()

## 2. Estadísticos y normalidad por variable

Se compara cada variable en su escala original y en logaritmo (el logaritmo solo aplica si todos los valores son positivos; la proporción de paradas tiene ceros).

In [ ]:
variables = ["dist_km", "moving_min", "elapsed_min", "elev_gain_m",
             "vel_media_kmh", "vel_max_kmh", "ratio_paradas"]
filas = []
for v in variables:
    x = m[v]
    positiva = (x > 0).all()
    filas.append({
        "variable": v,
        "media": x.mean(),
        "sd": x.std(),
        "asimetria": x.skew(),
        "curtosis_exceso": x.kurt(),
        "shapiro_p": stats.shapiro(x).pvalue,
        "asimetria_log": stats.skew(np.log(x)) if positiva else np.nan,
        "shapiro_p_log": stats.shapiro(np.log(x)).pvalue if positiva else np.nan,
    })
pd.DataFrame(filas)

**Lectura.** La distancia casi no varía (sd ≈ 0,37 km) y su asimetría (≈ −4,8) viene de dos salidas más cortas (11,0 y 12,1 km). Ninguna variable pasa Shapiro-Wilk con holgura, y el logaritmo no lo arregla: lo que rompe la normalidad son las salidas atípicas, no la forma de la escala. Eso es justo lo que el estimador robusto debe manejar.

In [ ]:
VARS_MODELO = ["dist_km", "elev_gain_m", "vel_media_kmh", "vel_max_kmh", "ratio_paradas"]
plots.distribuciones(m, VARS_MODELO)
plt.show()

## 3. Correlaciones y redundancia

La velocidad media es, por definición, distancia / tiempo en movimiento. Meter las tres en el mismo modelo introduce una dependencia exacta.

In [ ]:
VARS_CORR = ["dist_km", "moving_min", "elapsed_min", "elev_gain_m",
             "vel_media_kmh", "vel_max_kmh", "ratio_paradas"]
plots.correlacion(m, VARS_CORR)
plt.show()

error = (m["dist_km"] / (m["moving_min"] / 60) - m["vel_media_kmh"]).abs().max()
print(f"max |dist / tiempo - vel_media| = {error:.4f} km/h (solo redondeo)")

## 4. Condición de la matriz de covarianza

Mahalanobis necesita invertir Σ. Si las variables son linealmente dependientes, Σ es singular y la distancia no existe. Se mide con el menor autovalor de la matriz de correlación y su número de condición.

In [ ]:
def condicion(columnas, log):
    X = m[columnas].astype(float)
    if log:
        X = np.log(X)
    R = np.corrcoef(X.values, rowvar=False)
    return {"autovalor_min": np.linalg.eigvalsh(R).min(),
            "cond": np.linalg.cond(R)}

conjuntos = {
    "dist, tiempo, vel (en logs)": (["dist_km", "moving_min", "vel_media_kmh"], True),
    "dist, vel (en logs)": (["dist_km", "vel_media_kmh"], True),
    "VARS_MODELO (escala original)": (VARS_MODELO, False),
}
pd.DataFrame({k: condicion(*v) for k, v in conjuntos.items()}).T

**Lectura.** Con distancia, tiempo y velocidad en logaritmos, log(vel) = log(dist) − log(tiempo): el menor autovalor es ≈ 0 y el número de condición del orden de 10⁷. Σ es numéricamente singular. Por eso el conjunto de trabajo elimina `moving_min` y `elapsed_min` y se queda con **cinco variables** (`VARS_MODELO`) con condición ≈ 2–3.

## 5. Dependencia en el tiempo

Mahalanobis supone observaciones independientes. Las salidas están ordenadas en el tiempo; se mira cuánto se parece cada una a las anteriores.

In [ ]:
plots.serie_velocidad(m)
plt.show()

serie = m.sort_values("fecha")["vel_media_kmh"].values
for k in (1, 2, 3):
    print(f"autocorrelacion lag {k}: {np.corrcoef(serie[:-k], serie[k:])[0, 1]:.2f}")

**Lectura.** La autocorrelación en el lag 1 es ≈ 0,41: hay dependencia temporal. Además se ve un cambio de régimen: ≈ 20 km/h hasta diciembre y ≈ 17 km/h desde enero (el autor indica que se relajó). Una sola matriz Σ para todo el periodo mezcla dos regímenes; en el siguiente notebook se compara contra una versión por periodo.

## 6. Atípicos univariados (referencia)

Lo que detecta el criterio |z| > 3 variable por variable. Es la línea base que Mahalanobis debe superar: encontrar salidas raras *en conjunto* aunque ninguna variable lo sea por sí sola.

In [ ]:
fila = {}
for v in VARS_MODELO:
    marca = plots.marcar_z(m[v])
    fila[v] = list(m.loc[marca, "fecha"])
for v, fechas in fila.items():
    print(f"{v:15s} {len(fechas)} -> {fechas}")

## Decisiones para el siguiente notebook

1. Variables: `dist_km`, `elev_gain_m`, `vel_media_kmh`, `vel_max_kmh`, `ratio_paradas` (p = 5). Sin `moving_min` ni `elapsed_min`.
2. Sin transformar a log: no corrige la normalidad y `ratio_paradas` tiene ceros. Se revisa con `log1p` como análisis de sensibilidad.
3. Comparar Mahalanobis clásico vs. robusto (MCD), y Σ global vs. Σ por periodo.
4. `vel_max_kmh` tiene valores > 50 km/h que pueden ser saltos de GPS: hay que mirarlos, no asumir que son reales.
5. Declarar la dependencia temporal como limitación.